In [1]:
!pip install -q openai datasets tqdm pillow

In [2]:
import getpass
import os
import io
import json
import time
import base64
from tqdm import tqdm
from PIL import Image
from datasets import load_dataset
from openai import OpenAI

In [3]:
GEMINI_API_KEY = getpass.getpass("Nhập key vô")
client = OpenAI(
    api_key = GEMINI_API_KEY,
    base_url = "http://127.0.0.1:8317/v1"
)

Nhập key vô ········


In [4]:
MODEL_ID ="gemini-3.8-flash-high"

In [5]:
def encode_image_to_base64(image: Image.Image) -> str:
    buffered = io.BytesIO()
    if image.mode != "RGB":
        image = image.convert("RGB")
    image.save(buffered, format="JPEG")
    return base64.b64encode(buffered.getvalue()).decode("utf-8")

In [6]:
SYSTEM_INSTRUCTION = """Bạn là chuyên gia sư phạm hình học.
Nhiệm vụ của bạn là đọc đề bài, quan sát hình vẽ và giải thích chi tiết, chặt chẽ từng bước dẫn tới đáp án đã cho.
Quy định định dạng đầu ra:
1. Đặt toàn bộ quá trình lập luận logic, quan sát hình học, định lý áp dụng vào trong cặp thẻ <think> và </think>.
2. Kết luận đáp án cuối cùng ở dạng: The final answer is \\boxed{<đáp án>}."""

In [7]:
def generate_cot_solution(image: Image.Image, problem_text: str, ground_truth: str):
    base64_img = encode_image_to_base64(image)
    user_prompt = f"""Đề bài: {problem_text}
Đáp án đúng đã biết: {ground_truth}

Hãy phân tích hình vẽ và viết lời giải chi tiết từng bước chứng minh để đi đến đáp án trên theo đúng định dạng yêu cầu."""

    try:
        response = client.chat.completions.create(
            model = MODEL_ID,
            messages = [
                {"role": "system", "content": SYSTEM_INSTRUCTION},
                {
                    "role": "user",
                    "content": [
                        {"type": "text", "text": user_prompt},
                        {
                            "type": "image_url",
                            "image_url": {
                                "url": f"data:image/jpeg;base64,{base64_img}"
                            }
                        }
                    ]
                }
            ],
            temperature = 0.2
        )
        return response.choices[0].message.content
    except Exception as e:
        print(f"Lỗi khi gọi API: {e}")
        return None

In [8]:
print("Đang tải dataset...")
dataset = load_dataset("hiyouga/geometry3k", split="train")

output_file = "geometry3k_cot_sft.jsonl"
print(f"Bắt đầu chưng cất dữ liệu vào {output_file}...")

Đang tải dataset...
Bắt đầu chưng cất dữ liệu vào geometry3k_cot_sft.jsonl...


In [9]:
total_samples = len(dataset)
print(f"Tổng số bài toán cần chưng cất: {total_samples}")

Tổng số bài toán cần chưng cất: 2101


In [10]:
processed_ids = set()
if os.path.exists(output_file):
    with open(output_file, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                try:
                    data = json.loads(line)
                    processed_ids.add(data["id"])
                except Exception:
                    pass
print(f"Đã xử lý trước đó: {len(processed_ids)}/{total_samples} mẫu.")

Đã xử lý trước đó: 1897/2101 mẫu.


In [11]:
with open(output_file, "a", encoding="utf-8") as f:
    for idx, sample in enumerate(tqdm(dataset, desc="Đang chưng cất")):
        # Bỏ qua bài đã chưng cất thành công
        if idx in processed_ids:
            continue

        image = sample.get("images")[0]
        problem = sample.get("problem")
        answer = str(sample.get("answer"))

        solution = generate_cot_solution(image, problem, answer)

        if solution:
            record = {
                "id": idx,
                "problem": problem,
                "cot_solution": solution,
                "answer": answer
            }
            f.write(json.dumps(record, ensure_ascii=False) + "\n")
            f.flush()
            processed_ids.add(idx)

        # Tránh chạm ngưỡng 15 RPM của gói Free
        # time.sleep(4)

print("\nHoàn tất 100% quá trình chưng cất!")

Đang chưng cất: 100%|██████████| 2101/2101 [40:00<00:00,  1.14s/it] 


Hoàn tất 100% quá trình chưng cất!
